# 00 - TF-IDF + Cosine Similarity

In this notebook, I investigate how different **TF-IDF** approaches perform on the Semantic Textual Similarity Benchmark (STS-B) dataset.

I will implement and compare the following models:

1. Manual TF-IDF + Cosine Similarity
2. Manual TF-IDF without Stopwords
3. Scikit-learn TF-IDF
4. Scikit-learn TF-IDF with Unigrams and Bigrams
5. Scikit-learn TF-IDF with Sublinear TF

- The IDF values are calculated using only the training split of the STS-B dataset. This prevents data leakage and ensures that the validation and test sets remain unseen during model development.

- For each model, sentence representations are generated using TF-IDF and sentence similarity is calculated using cosine similarity. The predicted similarity scores are then compared with the human similarity scores provided in the dataset.

- To evaluate the models, I use Pearson and Spearman correlation coefficients.
    - **Pearson correlation** measures how closely the predicted similarity scores follow the human similarity scores. A higher Pearson value indicates that the model produces similarity scores that are more consistent with human judgements.
    - **Spearman correlation** measures how well the model ranks sentence pairs according to their similarity. This is useful because a model may not predict the exact human score but can still correctly rank more similar and less similar sentence pairs.


**The validation set** is used to compare the different TF-IDF configurations and identify the best-performing models. The top two models are then evaluated on the test set to measure their performance on unseen data.

**The main aim of these experiments** is to understand how different TF-IDF configurations affect semantic similarity performance before moving on to embedding-based methods such as GloVe and BERT.

In [87]:
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np
import pandas as pd
import math

from scipy.stats import pearsonr, spearmanr

from collections import Counter

from datasets import load_dataset


## Load Semantic Textual Similarity Benchmark (STS-B) Data

In [2]:
dataset = load_dataset("sentence-transformers/stsb")

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 1379
    })
})

In [4]:
print(dataset["train"][0])

{'sentence1': 'A plane is taking off.', 'sentence2': 'An air plane is taking off.', 'score': 1.0}


In [5]:
print(dataset["train"][1]["sentence1"])

A man is playing a large flute.


In [6]:
dataset["train"][:5]

{'sentence1': ['A plane is taking off.',
  'A man is playing a large flute.',
  'A man is spreading shreded cheese on a pizza.',
  'Three men are playing chess.',
  'A man is playing the cello.'],
 'sentence2': ['An air plane is taking off.',
  'A man is playing a flute.',
  'A man is spreading shredded cheese on an uncooked pizza.',
  'Two men are playing chess.',
  'A man seated is playing the cello.'],
 'score': [1.0, 0.76, 0.76, 0.52, 0.85]}

## Models

### 1. Manual TF-IDF + Cosine Similarity

This baseline manually builds a vocabulary from the training sentences, 
computes IDF values for each token, then represents each sentence as a TF-IDF vector. 
Similarity between two sentences is calculated using cosine similarity. 

This method is simple and interpretable because words with higher importance contribute more to the similarity score. 

However, it mainly captures lexical overlap, so it struggles when two sentences have the same meaning but use different words.

In [54]:
# Tokenized all sentences in training data
tokenized_sents = []

for sample in dataset["train"]:
    tokenized_sents.append([token.lower() for token in word_tokenize(sample["sentence1"]) if token.isalpha()])
    tokenized_sents.append([token.lower() for token in word_tokenize(sample["sentence2"]) if token.isalpha()])

In [55]:
tokenized_sents[:5]

[['a', 'plane', 'is', 'taking', 'off'],
 ['an', 'air', 'plane', 'is', 'taking', 'off'],
 ['a', 'man', 'is', 'playing', 'a', 'large', 'flute'],
 ['a', 'man', 'is', 'playing', 'a', 'flute'],
 ['a', 'man', 'is', 'spreading', 'shreded', 'cheese', 'on', 'a', 'pizza']]

In [56]:
len(tokenized_sents)

11498

In [57]:
# Create set of vocabulary
vocab = {token for sent in tokenized_sents for token in sent}

In [58]:
len(vocab)

10689

#### Compute IDF

In [59]:
# Number of documents
num_doc = len(tokenized_sents)

# To store IDF values
idf = {}

for token in vocab:

    # Calculate document frequency
    doc_freq = sum(1 for sent in tokenized_sents if token in sent)

    # Calculate idf for each token
    idf[token] = math.log((num_doc + 1) / (doc_freq  + 1)) # +1 is smoothing, ZeroDivisionError handle

In [60]:
idf

{'lettuce': 8.65686817348872,
 'atic': 8.65686817348872,
 'bristol': 7.963720992928774,
 'ashraf': 8.65686817348872,
 'pressed': 7.963720992928774,
 'swap': 8.65686817348872,
 'lot': 7.152790776712446,
 'reviewed': 8.65686817348872,
 'christened': 8.251403065380556,
 'night': 5.766496415592555,
 'note': 6.577426631808883,
 'received': 6.6419651529464545,
 'explicitly': 7.963720992928774,
 'spam': 8.251403065380556,
 'recess': 8.65686817348872,
 'm': 8.65686817348872,
 'drugs': 6.865108704260664,
 'macedonia': 8.251403065380556,
 'amnesty': 6.865108704260664,
 'sloot': 8.65686817348872,
 'palestinian': 5.175628084153027,
 'signature': 8.65686817348872,
 'draws': 8.251403065380556,
 'contests': 8.65686817348872,
 'registered': 7.963720992928774,
 'ronald': 8.65686817348872,
 'john': 5.794667292559251,
 'botswana': 8.251403065380556,
 'pagers': 8.65686817348872,
 'influence': 8.251403065380556,
 'die': 6.305492916325242,
 'spots': 7.963720992928774,
 'singled': 8.251403065380556,
 'desk':

#### Compute TF, TF-IDF and Cosine Similarity

In [64]:
# Define compute_tf fucntion to compute TF
def compute_tf(sent):

    # Lowercase tokens keep them in a list
    tokens= [token.lower() for token in word_tokenize(sent) if token.isalpha()]

    # Compute frequency of each word
    token_freq = FreqDist(tokens)

    tf = {} # to store tf values

    for token, freq in token_freq.items():

        # Compute TF
        tf[token] = freq / len(tokens)

    return tf

In [65]:
# Define compute_tfidf fucntion to compute TF-IDF
def compute_tfidf(sent, idf):

    # Compute TF 
    tf = compute_tf(sent)

    tfidf = {} # To store tf-idf values
    
    for token, tf_value in tf.items():

        # Ignore OOV tokens
        if token in idf:

            # Compute TF-IDF
            tfidf[token] = tf_value * idf[token]

    return tfidf

In [66]:
# Compute Cosine similarity between two TF-IDF vectors
def compute_cosine_similarity(vec1, vec2, vocab):

    # Convert dictionary vectors into ordered lists
    v1 = [vec1.get(token, 0) for token in vocab]
    v2 = [vec2.get(token, 0) for token in vocab]

    # Calculate the dot product of the two vectors
    dot_product = np.dot(v1, v2)

    # Calculate the magnitude of each vector
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)

    # Compute cosine similarity
    cos_sim = dot_product / (norm_v1 * norm_v2)

    return cos_sim

In [67]:
dataset["validation"]

Dataset({
    features: ['sentence1', 'sentence2', 'score'],
    num_rows: 1500
})

In [68]:
validation_df = []

for sample in dataset["validation"]:
    
    sent1 = sample["sentence1"]
    sent2 = sample["sentence2"]
    score = sample["score"]

    tfidf1 = compute_tfidf(sent1, idf)
    tfidf2 = compute_tfidf(sent2, idf)

    similarity = compute_cosine_similarity(tfidf1, tfidf2, vocab)

    validation_df.append({"sentence1": sent1,
                          "sentence2": sent2,
                          "human_score": score,
                          "pred_score": similarity})
    

In [69]:
validation_df = pd.DataFrame(validation_df)

validation_df.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.894361
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.880409
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,0.973533
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.819081
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.854326


In [70]:
manuel_pearson, _ = pearsonr(validation_df["pred_score"], validation_df["human_score"])
manuel_spearman, _ = spearmanr(validation_df["pred_score"], validation_df["human_score"])

print(f"Manual TF-IDF + Cosine Similarity, Pearson: {manuel_pearson:.4f}")
print(f"Manual TF-IDF + Cosine Similarity, Spearman: {manuel_spearman:.4f}")

Manual TF-IDF + Cosine Similarity, Pearson: 0.7026
Manual TF-IDF + Cosine Similarity, Spearman: 0.7018


The manual TF-IDF model achieved a `Pearson score of 0.7026` and a `Spearman score of 0.7018` on the validation set.

This result shows that TF-IDF can capture sentence similarity to a reasonable level using word overlap.
The close Pearson and Spearman scores indicate that the model is able to both predict similarity scores and rank sentence pairs consistently. 

However, the model is still limited because it only compares shared words. If two sentences have similar meaning but use different words, TF-IDF may give them a low similarity score.

### 2. Manual TF-IDF Without Stopwords

This version removes common English stopwords such as “the”, “is”, and “a” before building the TF-IDF representation. 

The goal is to reduce noise from frequent function words and focus more on meaningful content words.

In [106]:
stop_words = stopwords.words("english")

In [107]:
sents_out_stopwords = []

for sent in tokenized_sents:

    # Remove stopwords and punctuations
    sent_filtered = [token for token in sent if token not in stop_words]
    
    sents_out_stopwords.append(sent_filtered)

In [108]:
# Create new vocabulary without punctuation and stopwords
vocab_no_stopwords = {token for sent in sents_out_stopwords for token in sent}

In [109]:
len(vocab_no_stopwords)

10562

#### Compute new IDF for 'vocab_no_stopwords'

In [73]:
num_doc2 = len(sents_out_stopwords)

idf2 = {}

for token in vocab_no_stopwords:
    
    doc_freq = sum(1 for sent in sents_out_stopwords if token in sent)

    idf2[token] = math.log((num_doc2 + 1) / (doc_freq + 1))

In [74]:
def compute_tf2(sent):

    tokenized_sent = [token.lower() for token in word_tokenize(sent) if token.lower().isalpha() and token.lower() not in stop_words]
    token_freq = FreqDist(tokenized_sent)

    tf = {}
    for token, freq in token_freq.items():
        tf[token] = freq / len(tokenized_sent)

    return tf

In [75]:
def compute_tfidf2(sent, idf):

    tf = compute_tf2(sent)

    tfidf = {}
    
    for token, tf_value in tf.items():

        # Ignore OOV tokens
        if token in idf:
            tfidf[token] = tf_value * idf[token]

    return tfidf

In [76]:
validation_df_no_stopwords = []

for sample in dataset["validation"]:
    
    sent1 = sample["sentence1"]
    sent2 = sample["sentence2"]
    score = sample["score"]

    tfidf1 = compute_tfidf2(sent1, idf2)
    tfidf2 = compute_tfidf2(sent2, idf2)

    similarity = compute_cosine_similarity(tfidf1, tfidf2, vocab_no_stopwords)

    validation_df_no_stopwords.append({"sentence1": sent1,
                                       "sentence2": sent2,
                                       "human_score": score,
                                       "pred_score": similarity})

In [77]:
validation_df2 = pd.DataFrame(validation_df_no_stopwords)

validation_df2.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.913711
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.871020
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,1.000000
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.818149
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.862552


In [79]:
man_nostop_pearson, _ = pearsonr(validation_df2["pred_score"], validation_df2["human_score"])
man_nostop_spearman, _ = spearmanr(validation_df2["pred_score"], validation_df2["human_score"])


print(f"Manual TF-IDF Without Stopwords, Pearson: {man_nostop_pearson:.4f}")
print(f"Manual TF-IDF Without Stopwords, Spearman: {man_nostop_spearman:.4f}")


Manual TF-IDF Without Stopwords, Pearson: 0.7068
Manual TF-IDF Without Stopwords, Spearman: 0.7104


The manual TF-IDF model without stopwords achieved `Pearson score of 0.7068` and `Spearman score of 0.7104` on the validation set.

Compared to the baseline manual TF-IDF model `Pearson = 0.7026, Spearman = 0.7018`, removing stopwords improved `Pearson score by 0.0042` and `Spearman score by 0.0086`.

This suggests that common words such as "the", "a", and "is" contribute little to measuring sentence similarity and may introduce noise into the TF-IDF representation.

Overall, the results indicate that removing stopwords helps the model focus more on meaningful content words, leading to slightly better performance.

### 3. Scikit-learn TF-IDF

This version uses `TfidfVectorizer` from scikit-learn with English stopword removal. 

It is cleaner, faster, and more reliable than the manual implementation because it handles preprocessing, IDF weighting, vector normalization, and sparse matrix operations efficiently. 

In [30]:
train_sentences = []

for sample in dataset["train"]:
    train_sentences.append(sample["sentence1"])
    train_sentences.append(sample["sentence2"])

vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")

vectorizer.fit(train_sentences)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.

In [31]:
validation_df_sci = []

for sample in dataset["validation"]:
    
    vec1 = vectorizer.transform([sample["sentence1"]])
    vec2 = vectorizer.transform([sample["sentence2"]])

    similarity = cosine_similarity(vec1, vec2)[0][0]

    validation_df_sci.append({"sentence1": sample["sentence1"],
                               "sentence2": sample["sentence2"],
                               "human_score": sample["score"],
                               "pred_score": similarity})

In [32]:
validation_df3 = pd.DataFrame(validation_df_sci)

validation_df3.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.907134
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.870008
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,1.000000
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.787308
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.828134


In [88]:
sci_pearson, _ = pearsonr(validation_df3["pred_score"], validation_df3["human_score"])
sci_spearman, _ = spearmanr(validation_df3["pred_score"], validation_df3["human_score"])


print(f"Scikit-learn TF-IDF, Pearson: {sci_pearson:.4f}")
print(f"Scikit-learn TF-IDF, Spearman: {sci_spearman:.4f}")

Scikit-learn TF-IDF, Pearson: 0.7258
Scikit-learn TF-IDF, Spearman: 0.7276


The Scikit-learn TF-IDF model achieved `Pearson score of 0.7258` and `Spearman score of 0.7276` on the validation set.

Compared to the manual TF-IDF model `Pearson = 0.7026, Spearman = 0.7018`, the Scikit-learn implementation improved the `Pearson score by 0.0232` and the `Spearman score by 0.0258`.

It also outperformed the manual TF-IDF model without stopwords `Pearson = 0.7068, Spearman = 0.7104`, achieving an improvement of `0.019 in Pearson` and `0.0172 in Spearman`.

Overall, the results suggest that the Scikit-learn implementation produces a more effective TF-IDF representation for measuring sentence similarity on the STS-B dataset.

### 4. Scikit-learn TF-IDF with Unigrams and Bigrams

This model uses both single words and two-word phrases with `ngram_range=(1, 2)`. 

The aim is to capture short phrases such as “playing guitar” or “hard hat”, not just isolated words.

In [34]:
vectorizer2 = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2))

vectorizer2.fit(train_sentences)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.

In [35]:
validation_df_sci2 = []

for sample in dataset["validation"]:
    
    vec1 = vectorizer2.transform([sample["sentence1"]])
    vec2 = vectorizer2.transform([sample["sentence2"]])

    similarity = cosine_similarity(vec1, vec2)[0][0]

    validation_df_sci2.append({"sentence1": sample["sentence1"],
                               "sentence2": sample["sentence2"],
                               "human_score": sample["score"],
                               "pred_score": similarity})

In [36]:
validation_df4 = pd.DataFrame(validation_df_sci2)

validation_df4.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.763102
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.745688
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,1.000000
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.606828
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.678414


In [89]:
sci_ngram_pearson, _ = pearsonr(validation_df4["pred_score"], validation_df4["human_score"])
sci_ngram_spearman, _ = spearmanr(validation_df4["pred_score"], validation_df4["human_score"])


print(f"Scikit-learn TF-IDF with Unigrams and Bigrams, Pearson: {sci_ngram_pearson:.4f}")
print(f"Scikit-learn TF-IDF with Unigrams and Bigrams, Spearman: {sci_ngram_spearman:.4f}")

Scikit-learn TF-IDF with Unigrams and Bigrams, Pearson: 0.6992
Scikit-learn TF-IDF with Unigrams and Bigrams, Spearman: 0.7112


The Scikit-learn TF-IDF model with unigrams and bigrams achieved `Pearson score of 0.6992` and `Spearman score of 0.7112` on the validation set.

Compared to the standard Scikit-learn TF-IDF model `Pearson = 0.7258, Spearman = 0.7276`, adding bigrams reduced `Pearson score by 0.0266` and `Spearman score by 0.0164`.

This suggests that including bigrams did not improve sentence similarity performance on the STS-B dataset. 
A possible reason is that bigrams create a larger and more sparse feature space, reducing the overlap between semantically similar sentences.

Overall, the unigram TF-IDF model performed better than the unigram and bigram combination for this task.

### 5. Scikit-learn TF-IDF with Sublinear TF


This version uses sublinear_tf=True, which replaces raw term frequency with logarithmic scaling.

As a result, repeated words contribute less to the final TF-IDF score. 

The aim is to reduce the influence of words that appear many times within the same document and place more emphasis on whether a word appears rather than how often it appears.

In [38]:
vectorizer3 = TfidfVectorizer(lowercase=True, stop_words="english", sublinear_tf=True)

vectorizer3.fit(train_sentences)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.

In [39]:
validation_df_sci3 = []

for sample in dataset["validation"]:
    
    vec1 = vectorizer3.transform([sample["sentence1"]])
    vec2 = vectorizer3.transform([sample["sentence2"]])

    similarity = cosine_similarity(vec1, vec2)[0][0]

    validation_df_sci3.append({"sentence1": sample["sentence1"],
                               "sentence2": sample["sentence2"],
                               "human_score": sample["score"],
                               "pred_score": similarity})

In [40]:
validation_df5 = pd.DataFrame(validation_df_sci3)

validation_df5.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.907134
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.870008
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,1.000000
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.787308
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.828134


In [90]:
sublienar_pearson, _ = pearsonr(validation_df5["pred_score"], validation_df5["human_score"])
sublienar_spearman, _ = spearmanr(validation_df5["pred_score"], validation_df5["human_score"])


print(f"Scikit-learn TF-IDF with Sublinear TF, Pearson: {sublienar_pearson:.4f}")
print(f"Scikit-learn TF-IDF with Sublinear TF, Spearman: {sublienar_spearman:.4f}")

Scikit-learn TF-IDF with Sublinear TF, Pearson: 0.7262
Scikit-learn TF-IDF with Sublinear TF, Spearman: 0.7277


The Scikit-learn TF-IDF model with sublinear TF achieved `Pearson score of 0.726` and `Spearman score of 0.728` on the validation set.

Compared to the standard Scikit-learn TF-IDF model `Pearson = 0.726, Spearman = 0.728`, the differences were very small. The `Pearson score increased by 0.0004` and `Spearman score increased by 0.0001`.

These results suggest that applying sublinear TF had almost no impact on performance. 

One possible reason is that the STS-B dataset contains relatively short sentences, so repeated words may not occur often enough for TF scaling to make a noticeable difference. 

Overall, the sublinear TF model performed almost identically to the standard Scikit-learn TF-IDF model.

In [121]:
repeated_tokens = 0

total_tokens = 0

for sample in dataset["train"]:

    for sent in [sample["sentence1"], sample["sentence2"]]:

        tokens = [token.lower() for token in word_tokenize(sent) if token.isalpha() and token.lower() not in stop_words]

        counts = Counter(tokens)

        for count in counts.values():

            total_tokens += 1

            if count > 1:
                repeated_tokens += 1

print(f"Percentage of unique words repeated: {(repeated_tokens / total_tokens):.3%}")
      

Percentage of unique words repeated: 1.004%


Only `1.004%` of unique words appeared more than once after preprocessing. 

This suggests that repeated words are rare in the dataset, which may explain why sublinear TF produced almost no improvement.

## Validation comparison across models

In [101]:
comparison_df = pd.DataFrame([["Manual TF-IDF", manuel_pearson, manuel_spearman],
                              ["Manual TF-IDF No Stopwords", man_nostop_pearson, man_nostop_spearman],
                              ["Scikit TF-IDF", sci_pearson, sci_spearman],
                              ["Scikit TF-IDF + Bigrams", sci_ngram_pearson, sci_ngram_spearman],
                              ["Scikit TF-IDF + Sublinear TF", sublienar_pearson, sublienar_spearman]], 
                              columns=["Model", "Pearson", "Spearman"]).round(4)

comparison_df

,Model,Pearson,Spearman
0,Manual TF-IDF,0.7026,0.7018
1,Manual TF-IDF No Stopwords,0.7068,0.7104
2,Scikit TF-IDF,0.7258,0.7276
3,Scikit TF-IDF + Bigrams,0.6992,0.7112
4,Scikit TF-IDF + Sublinear TF,0.7262,0.7277


The validation results show that the **Scikit-learn TF-IDF models** performed better than the manual implementations. 

The highest scores were achieved by **Scikit-learn TF-IDF with sublinear TF** `Pearson = 0.726, Spearman = 0.728`, but the improvement over the **standard Scikit-learn TF-IDF** model `Pearson = 0.726, Spearman = 0.728` was very small.

The manual TF-IDF model without stopwords performed slightly better than the baseline manual TF-IDF model, improving `Pearson score from 0.7026 to 0.7068` and `Spearman score from 0.7018 to 0.7104`.

Adding bigrams reduced performance, with `Pearson score decreasing from 0.7258 to 0.6992`. This suggests that adding more features does not always improve sentence similarity performance.

Based on the validation results, the two best-performing models are **Scikit-learn TF-IDF** and **Scikit-learn TF-IDF with sublinear TF**. 

These models will be evaluated on the test set in the next section.

## Test Set Evaluation of the Top Two Models

Before evaluating the final models on the test set, an Out-of-Vocabulary (OOV) analysis will be carried out.

OOV words are words that appear in the test set but were not seen in the training data. Since TF-IDF can only assign weights to words that exist in its vocabulary, OOV words do not contribute to the sentence representation and may affect performance.

In [111]:
oov_tokens = 0
total_tokens = 0

for sample in dataset["test"]:

    for sent in [sample["sentence1"], sample["sentence2"]]:

        tokens = [token.lower() for token in word_tokenize(sent) if token.isalpha() and token.lower() not in stop_words]

        for token in tokens:

            total_tokens += 1

            if token not in vocab_no_stopwords:
                oov_tokens += 1

print(f"Test OOV rate: {(oov_tokens / total_tokens):.3%}")

Test OOV rate: 9.910%


The OOV analysis showed a test OOV rate of `9.91%`. 

This means that almost 10% of the words in the test set were not present in the training vocabulary after preprocessing. 

As a result, these words could not contribute to the TF-IDF vectors used to calculate sentence similarity.

### A. Scikit-learn TF-IDF

In [113]:
test_df_sci = []

for sample in dataset["test"]:
    
    vec1 = vectorizer.transform([sample["sentence1"]])
    vec2 = vectorizer.transform([sample["sentence2"]])

    similarity = cosine_similarity(vec1, vec2)[0][0]

    test_df_sci.append({"sentence1": sample["sentence1"],
                        "sentence2": sample["sentence2"],
                        "human_score": sample["score"],
                        "pred_score": similarity})

In [114]:
test_df = pd.DataFrame(test_df_sci)

test_df.head()

,sentence1,sentence2,human_score,pred_score
0,A girl is styling her hair.,A girl is brushing her hair.,0.50,0.489599
1,A group of men play soccer on the beach.,A group of boys are playing soccer on the beach.,0.72,0.616470
2,One woman is measuring another woman's ankle.,A woman measures another woman's ankle.,1.00,0.690376
3,A man is cutting up a cucumber.,A man is slicing a cucumber.,0.84,0.698280
4,A man is playing a harp.,A man is playing a keyboard.,0.30,0.324109


In [115]:
test_sci_pearson, _ = pearsonr(test_df["pred_score"], test_df["human_score"])
test_sci_spearman, _ = spearmanr(test_df["pred_score"], test_df["human_score"])


print(f"Scikit-learn TF-IDF, Pearson: {test_sci_pearson:.4f}")
print(f"Scikit-learn TF-IDF, Spearman: {test_sci_spearman:.4f}")


Scikit-learn TF-IDF, Pearson: 0.6292
Scikit-learn TF-IDF, Spearman: 0.6137


### B. Scikit-learn TF-IDF with Sublinear TF

In [116]:
test_df_sci2 = []

for sample in dataset["test"]:
    
    vec1 = vectorizer3.transform([sample["sentence1"]])
    vec2 = vectorizer3.transform([sample["sentence2"]])

    similarity = cosine_similarity(vec1, vec2)[0][0]

    test_df_sci2.append({"sentence1": sample["sentence1"],
                         "sentence2": sample["sentence2"],
                         "human_score": sample["score"],
                         "pred_score": similarity})

In [117]:
test_df2 = pd.DataFrame(test_df_sci2)

test_df2.head()

,sentence1,sentence2,human_score,pred_score
0,A girl is styling her hair.,A girl is brushing her hair.,0.50,0.489599
1,A group of men play soccer on the beach.,A group of boys are playing soccer on the beach.,0.72,0.616470
2,One woman is measuring another woman's ankle.,A woman measures another woman's ankle.,1.00,0.628421
3,A man is cutting up a cucumber.,A man is slicing a cucumber.,0.84,0.698280
4,A man is playing a harp.,A man is playing a keyboard.,0.30,0.324109


In [119]:
test_sublienar_pearson, _ = pearsonr(test_df2["pred_score"], test_df2["human_score"])
test_sublienar_spearman, _ = spearmanr(test_df2["pred_score"], test_df2["human_score"])


print(f"Scikit-learn TF-IDF with Sublinear TF, Pearson: {test_sublienar_pearson:.4f}")
print(f"Scikit-learn TF-IDF with Sublinear TF, Spearman: {test_sublienar_spearman:.4f}")

Scikit-learn TF-IDF with Sublinear TF, Pearson: 0.6304
Scikit-learn TF-IDF with Sublinear TF, Spearman: 0.6149


**The Scikit-learn TF-IDF** model achieved `Pearson score of 0.629` and `Spearman score of 0.614` on the test set. 

**The TF-IDF model with sublinear TF** achieved slightly higher scores, with `Pearson score of 0.630` and `Spearman score of 0.615`.

The difference between the two models is very small, which is consistent with the validation results. Although the sublinear TF model achieved the highest scores, the improvement was only 0.001 in both Pearson and Spearman correlation.

Compared to the validation results, both models achieved lower scores on the test set. This shows that **the models generalised less effectively to unseen data**. 

The OOV analysis showed that `9.91%` of test words were not present in the training vocabulary and therefore could not contribute to the TF-IDF representation. This may have contributed to the lower test performance.

Overall, the results show that TF-IDF provides a reasonable baseline for semantic similarity, but its performance is limited because it mainly relies on shared words rather than understanding sentence meaning.

## Explanation of WHY TF-IDF works and WHERE it fails

TF-IDF works by giving higher weights to important words and lower weights to very common words. 
If two sentences share important words, their TF-IDF vectors become more similar, which usually results in a higher cosine similarity score.

The results in this notebook show that TF-IDF can be used as a reasonable baseline for semantic similarity. 
The best TF-IDF model achieved a Pearson score of `0.630 on the test set`, showing that word overlap can capture part of the similarity between two sentences.

However, TF-IDF has some important limitations. It depends on **exact word matching** and **does not understand the meaning of words**. 
For example, words such as "car" and "automobile" are treated as different features even though they have a similar meaning.

Another limitation is that TF-IDF **cannot capture context**. Two sentences may express the same idea using different words, 
but TF-IDF may still assign a low similarity score because there is little word overlap between them.

These limitations are one of the main reasons why word embeddings and transformer-based models have become popular for semantic similarity tasks.